# Analysis of constant acceleration MPC, for the C++ implementation

In [ ]:
%matplotlib ipympl

import functools

import h5py
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm

from exp_mpc.stewart_min import mpc_spec, opt, viz

jax.config.update("jax_enable_x64", True)

In [ ]:
# general setup (should conform with `mpc_export.py`)
limits = mpc_spec.MPCLimits()
spec = mpc_spec.MPCSpec()
data_ref = np.array(pd.read_hdf("../../data/cpp_const.hdf"))
acc_ref = data_ref[:, :3]
omega_ref = data_ref[:, 3:]

## visualize results

In [ ]:
with h5py.File("../../cpp/data/mpc_example_data.h5", "r") as f:
    control_res = np.array(f["control_res"])
    timings  = np.array(f["timings"]) * 1e-6

train_state = opt.TrainState.zero_init(spec, acc_ref[0, 2])
train_list = []
train_step = functools.partial(opt.train_step_with_cost, spec, opt_scheme="none")
for i in tqdm.tqdm(range(control_res.shape[0])):
    train_state.control = control_res[i]
    train_state, _, _ = train_step(train_state, acc_ref[i], omega_ref[i])
    train_list.append(train_state)

In [ ]:
freqs = 1.0 / np.array(timings)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(freqs)
ax.set_title("Optimization Frequency Over Time")
ax.set_xlabel("Iteration")
ax.set_ylabel("Frequency (Hz)")
ax.grid(True)
plt.show()

In [ ]:
sol_list_end = []
extra_steps = 0

tl = train_list
references= {
    "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(tl), 1)),
    "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(tl), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=tl, limits=limits, spec=spec, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=tl, limits=limits, spec=spec, use_estop=True)

## visualize timings

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(np.array(timings) * 1e3)
fig.suptitle("C++ timings")
ax.set_xlabel("iter")
ax.set_ylabel("time (ms)")
ax.set_ylim(2, 10)
ax.grid()
plt.show()

## animation

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
# mp_mpl.call_mp_animate_trajectory(
#     file_name="data/cpp_const_acc_3d.mp4",
#     trajectory=trajectory,
# )